In [0]:
# IMPORTANT:  
#
#   The methods to read files were modified to read/write in volume instead of s3 for DEV and QA. 
#   For example, if you go and execute a job read in DEV environment from:
#                         's3://memberanalytics-data-out-prod/'            +  'ASSIGNMENTS/cdsa/assn_output/PROD/MMPC19FY26_PROD/final_2025-11-11/mail_subset`
#   it will go look at:   '/Volumes/datascience_ea_dev/pe/outputs_for_s3/' +  'ASSIGNMENTS/cdsa/assn_output/PROD/MMPC19FY26_PROD/final_2025-11-11/mail_subset'`

#   The files in volume should be copied from s3 before running the process. 


# LL Note:  VERSION_MAP is currently not used given the yml config, check if this need to be migrated to dbx source
#           `VERSION_MAP: 's3://memberanalytics-data-out-prod/ASSIGNMENTS/ campaigns/FY21/BBM12FY21/Coupon_Input/version map bbm12.csv'`

In [0]:
# %load_ext autoreload
# %autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
%run ../../config/utils 

In [ ]:
# Databricks Runtime 12+ defaults to ansi.enabled=true which makes cast() throw errors
spark.conf.set("spark.sql.ansi.enabled", "false")

In [0]:
package_path = f"/Volumes/{catalog_name}/pe/helpers/packages/xlsxwriter/"

In [0]:
%pip install --no-index --find-links {package_path} xlsxwriter
dbutils.library.restartPython()

In [0]:
# checkpoint() requires a checkpoint directory — this MUST succeed
try:
    spark.sparkContext.setCheckpointDir("/dbfs/tmp/qc_assignments")
except Exception:
    spark.sparkContext.setCheckpointDir("/tmp/qc_assignments")
    print("Using /tmp/qc_assignments as checkpoint dir (shared cluster fallback)")

In [0]:
%run ../../config/utils

In [ ]:
# Re-set after restartPython() — the previous spark session config is lost
spark.conf.set("spark.sql.ansi.enabled", "false")

In [0]:
"""Spark job to QC an assignment run and generate a QC report for inspection."""
import sys
sys.path.append("..")
sys.path.append("../..")

from lib_assignment.assn_io     import JobManager, move_to_outbound, move_to_outbound_dbx
from lib_assignment.assn_utils  import has_coupons, subset_by_time, env_path
from lib_assignment.checks      import check_execution_overwrite
from lib_assignment.excelreport import write_report_dbx
from lib_assignment.qc          import *
from lib_assignment.qctests     import QCTestRunner

In [0]:
# Params 'output_vol' and 'environment' are configured in variables when this is exec: %run ../../config/utils

job = JobManager("qc", "../config/config_template.yml", spark, output_vol, environment) # "assignment QC"


job.config.params = dict(
    list(job.config.params.items())
    + list(job.config.cnf["sizing"].items())
    + list(job.config.cnf["subset"].items())
)

if job.config.params["run_type"].lower() == "prod":
    # avoid overwritting an existing process output
    check_execution_overwrite(
        job,
        paths_to_check=[
            job.config.paths["QC_REPORT"],
            job.config.paths["SAVINGS"],
        ]
    )

# Replace those inputs that can be pulled from the Databricks migrated sources:
job.config.paths['RAW_MEMBER']          = silver_master_member_extended
job.config.paths['CUBE']                = fs_customer_cube_full
job.config.paths['EXCLUSIONS']          = dna_brand_exclusions  # 's3://memberanalytics-data-out-prod/exclusions/exclusions_with_brand.csv'
job.config.paths['ITEM_MASTER']         = silver_master_item    # 's3://memberanalytics-data-out-prod/pipelined_intermediates/master/item'
job.config.paths['ARTICLE_AH4_AH5_MAP'] = silver_master_item    # yes, same as item_master
job.config.paths['ARTICLE_DNA_PATH']    = fs_article_nbr_category_dna_full # '/CATEGORY_DNA/PROD/ARTICLE_NBR/CATEGORY_DNA_full/PARQUET' 
job.config.paths['PRED_LIST']           = cf_prediction
job.config.paths['PROPENSITY']          = trip_spend_prediction

In [0]:
# basic job params
TYPE = int(
    "BBM" in job.config.params["campaign"].upper()
)  # 1 for BBM, 0 for other


EXAMPLES = [
    {"name": "Rong", "id": 58564596},
    {"name": "Doug", "id": 41510081},
    {"name": "Susan", "id": 76882031},
    {"name": "Keith", "id": 37837336},
    {"name": "Kristy", "id": 5285455},
    {"name": "Tom", "id": 9665302},
    {"name": "MEGHAN JOLIE", "id": 68770944},
    {"name": "SAMANTHA MANZELLO", "id": 61620238},
    {"name": "SONYA MCCORMACK", "id": 61842674},
    {"name": "Kelsey Gainor", "id": 70558415},
    {"name": "Michelle Crockford", "id": 58432200},
    {"name": "Match 1", "id": 2836659},
    {"name": "Match 2", "id": 4003915},
    {"name": "Match 3", "id": 4774762},
    {"name": "Match 4", "id": 33352891},
    {"name": "UnMatch 1", "id": 2390325},
    {"name": "UnMatch 2", "id": 7377514},
    {"name": "UnMatch 3", "id": 8248476},
    {"name": "UnMatch 4", "id": 40285602}
]

TESTS_TO_RUN = [
    "avg_spend_decr",
    "duplicate_coupons",
    "duplicate_members",
    "basket_cell",
    "trial_cell",
    "same_num_coups",
    "downsampled_coupons",
    "check_cell_size",
    "check_sensitive_content",
    "count_cf_ineligible", # keep last in list
]


# getting the downsampling information
job.config.params = dict(
    list(job.config.params.items())
    + list(job.config.cnf["assignment"].items())
)

In [0]:
print("Reading data...")


job.data.read("raw_member",     job.config.paths['RAW_MEMBER'],         filetype="table")
job.data.read("dna",            job.config.paths['CUBE'],               filetype="table")
job.data.read("item_master",    job.config.paths['ITEM_MASTER'],        filetype="table")
job.data.read("article_map",    job.config.paths['ARTICLE_AH4_AH5_MAP'],filetype="table")
job.data.read("article_dna",    job.config.paths['ARTICLE_DNA_PATH'],   filetype="table")  
job.data.read("cf",             job.config.paths['PRED_LIST'] ,         filetype="table")

# safety casting of malformed data 
job.data.tables['item_master'] = job.data.tables['item_master'].withColumn('ARTICLE_NBR', f.col('ARTICLE_NBR').try_cast('BIGINT'))
job.data.tables['article_map'] = job.data.tables['article_map'].withColumn('ARTICLE_NBR', f.col('ARTICLE_NBR').try_cast('BIGINT'))
job.data.tables['article_dna'] = job.data.tables['article_dna'].withColumn('ARTICLE_NBR', f.col('ARTICLE_NBR').try_cast('BIGINT'))


if job.config.paths["EXCLUSIONS"]:
    job.data.read("exclusion_rules", job.config.paths['EXCLUSIONS'] ,   filetype="table") 


job.data.tables["article_map"] = job.data.tables["article_map"].dropDuplicates(["article_nbr"])
job.data.tables["dna"] = subset_by_time(
    job.data.tables["dna"],
    job.config.params["assignment_date"],
    "fiscal_week",
)

job.data.read("mail_list",          "MAIL_LIST",            filetype="csv")
job.data.read("final_mailhouse",    "INPUT_MAILHOUSE",      filetype="csv") # auto --> es la salida de generate_output_file, deberiamos poder levantarlo del volumen o tabla. 

# Auto generated paths:
job.data.read("assignment",         "MAIL_POPULATION_ASSIGNMENT",   filetype="csv")
job.data.read("input_assignment",   "INPUT_ASSIGNMENTS",    filetype="csv")
job.data.read("constructs",         "INPUT_CONSTRUCTS",     filetype="parquet")
job.data.read("cell",               "CELL",                 filetype="csv", schema=job.data.schemas["cells"])
job.data.read("campaign",           "CAMPAIGN",             filetype="csv", schema=job.data.schemas["campaigns"],)
job.data.read("memtrips",           "COUPON_MEMTRIP",       filetype="parquet")
job.data.read("quals",              "COUPON_QUALS",         filetype="csv", schema=job.data.schemas["coupon_quals"],)
job.data.read("coups",              "COUPON_BANK",          filetype="csv", schema=job.data.schemas["coupon_bank"],)
job.data.read("coup_map",           "COUPON_MAP",           filetype="csv", schema=job.data.schemas["coupon_map"],)
job.data.read("coupon_bank",        "COUPON_BANK",          filetype="csv")
job.data.read("cdsa_assgn",         "CDSA_ASSGN",           filetype="parquet")


job.data.read("cpg_coupon",         "CPG_COUPON_LIST_PATH", filetype="csv", required=False)
job.data.read("category_coupon",    "CATEGORY_COUPON_PATH", filetype="csv", required=False,) # commented (missing like version_map)
job.data.read("basket_coupon",      "BASKET_COUPON_PATH",   filetype="csv", required=False)  # commented (missing like version_map)

if TYPE == 1 and job.config.paths.get("VERSION_MAP") is not None:
    job.data.read("version_map", "VERSION_MAP", filetype="csv") # LL Note:  currently not used, check if this need to be migrated to dbx source

In [0]:
print("Create core datasets")

generate_full_basedata(job, TYPE)  # requires base, coups, quals, preds, and memtrips

In [0]:
mailfile_to_savings(job)

print("aggregating...")

configured_tabs = generate_configured_qc(job)

In [0]:
generate_coupon_articles(job)

generate_coupon_dataset(job)  # requires full base and articles

generate_member_dataset(job)  # requires full base

generate_nomail_dataset(job) 

generate_decile_by_mail_dataset(job)  # requires full base and cell

In [0]:
generate_group_count_dataset(
    job,
    "slot-cpn",
    ["SLOT_NBR", "CPN_TYPE"],
    [sqlf.col("CPN_TYPE"), sqlf.col("SLOT_NBR")],
)  # requires full base


generate_group_count_dataset(
    job,
    "cell-cpn",
    ["CELL_ID", "CPN_TYPE"],
    [sqlf.col("CELL_ID"), sqlf.col("CPN_TYPE")],
)  # requires full base

In [0]:
is_long = generate_longitudinal_dataset(job)

EXAMPLES = generate_example_data(job, EXAMPLES)  # requires full base

In [0]:
print("running tests...") #  2025/11/04 it took 1 hs 11 min

runner = QCTestRunner(job, TESTS_TO_RUN)

testdata = runner.run_tests()

job.data.add("tests", testdata)

In [0]:

if is_long:
    output_tables = [
        "tests",
        "nomail_check",
        "decile_mail",
        "longitudinal",
        EXAMPLES,
        "coupon",
        "articles",
        "cell-cpn",
        "slot-cpn",
    ]
else:
    output_tables = [
        "tests",
        "nomail_check",
        "decile_mail",
        EXAMPLES,
        "coupon",
        "articles", 
        "cell-cpn",
        "slot-cpn",
    ]


    
compare_tabs = [ct for ct in configured_tabs if "compare" in ct]
configured_tabs = [c for c in configured_tabs if c not in compare_tabs]
temp = [output_tables[0]]
temp.extend(compare_tabs)
temp.extend(output_tables[1:])
output_tables = temp
output_tables.extend(configured_tabs)

In [0]:
print("writing QC report...")
write_report_dbx(job, output_tables, dbutils)

In [0]:
if job.config.params["run_type"].lower() == "prod":

    report_path = env_path(job.config.paths["QC_REPORT"], job.data.vol_base, job.data.env)

    # if environment == "prod":
    #     move_to_outbound(
    #         file_path=report_path, 
    #         file_name="qc", 
    #         file_type="xlsx", 
    #         params=job.config.params,
    #         paths=job.config.paths,
    #     )
    # else:
    move_to_outbound_dbx(
        file_path=report_path, 
        file_name="qc", 
        file_type="xlsx", 
        params=job.config.params,
        paths=job.config.paths,
        spark=spark, 
        vol_base=job.data.vol_base, 
        env=job.data.env, 
        dbutils=dbutils
    )